In [1]:
!pip install pycryptodome ecdsa base58 numpy torch

import sys
import hashlib
from Crypto.Hash import RIPEMD160
from ecdsa import SigningKey, SECP256k1
import base58
import numpy as np
import torch
import random
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
TARGET_HASH160_BYTES = bytes.fromhex("a0b0d60e5991578ed37cbda2b17d8b2ce23ab295")
BASE_PRIV = "8b2aca2022fdaffad19df9bfc576871af60841ef526f9a7b5fcfb5923ab98f8d"
BASE_HASH160 = bytes.fromhex("a0a8d608596e57bdfd53502ead468974df73f63c")

def derive_address_batch(priv_batch):
    results = []
    for priv_bytes in priv_batch:
        sk = SigningKey.from_string(priv_bytes, curve=SECP256k1)
        pub = b'\04' + sk.get_verifying_key().to_string()
        sha = hashlib.sha256(pub).digest()
        ripemd = RIPEMD160.new()
        ripemd.update(sha)
        ripemd_digest = ripemd.digest()
        payload = b'\x00' + ripemd_digest
        checksum = hashlib.sha256(hashlib.sha256(payload).digest()).digest()[:4]
        addr = base58.b58encode(payload + checksum).decode()
        results.append((addr, ripemd_digest.hex()))
    return results

def hamming_distance(a, b):
    return np.sum(np.frombuffer(a, dtype=np.uint8) != np.frombuffer(b, dtype=np.uint8))

def flip_bit_batch(base_priv, flipped_bits_batch):
    priv_batch = []
    for flipped_bits in flipped_bits_batch:
        priv_array = bytearray(base_priv)
        for bit_pos in flipped_bits:
            byte_idx = bit_pos // 8
            bit_idx = bit_pos % 8
            priv_array[byte_idx] ^= (1 << bit_idx)
        priv_batch.append(bytes(priv_array))
    return priv_batch

def get_hash160_diff_bits(base_h160, target_h160):
    diff_bits = []
    for byte_idx, (b, t) in enumerate(zip(base_h160, target_h160)):
        xor = b ^ t
        for bit_idx in range(8):
            if xor & (1 << bit_idx):
                diff_bits.append(byte_idx * 8 + bit_idx)
    return diff_bits[:16]

class BitflipAI:
    def __init__(self, base_priv, diff_bits):
        self.base_priv = bytes.fromhex(base_priv)
        self.diff_bits = diff_bits
        self.best_hamming = 16
        self.best_priv = base_priv

    def get_action(self, state):
        actions = [0] * 256
        for bit in self.diff_bits:
            if random.random() < 0.8:
                actions[bit] = 1
        for bit in range(256):
            if random.random() < 0.1:
                actions[bit] = 1 if actions[bit] == 0 else 0
        return actions

    def run(self):
        sys.stdout.write("*** Starting run ***\n")
        sys.stdout.flush()
        diff_bits = get_hash160_diff_bits(BASE_HASH160, TARGET_HASH160_BYTES)
        sys.stdout.write(f"*** Diff_bits: {diff_bits} ***\n")
        sys.stdout.flush()

        iteration = 0
        while iteration < 3:  # Hard limit
            iteration += 1
            state = bytes.fromhex(self.best_priv)
            sys.stdout.write(f"*** Iter {iteration} - State: {state.hex()[:16]}... ***\n")
            sys.stdout.flush()

            actions_batch = [self.get_action(state) for _ in range(5)]  # Tiny batch
            flipped_bits_batch = [[i for i, a in enumerate(actions) if a == 1] for actions in actions_batch]
            sys.stdout.write(f"*** Iter {iteration} - Flipped bits: {flipped_bits_batch} ***\n")
            sys.stdout.flush()

            priv_batch = flip_bit_batch(state, flipped_bits_batch)
            sys.stdout.write(f"*** Iter {iteration} - Privs: {[p.hex()[:16] for p in priv_batch]} ***\n")
            sys.stdout.flush()

            addr_hash160_batch = derive_address_batch(priv_batch)
            for i, (addr, hash160) in enumerate(addr_hash160_batch):
                hamm = hamming_distance(bytes.fromhex(hash160), TARGET_HASH160_BYTES)
                sys.stdout.write(f"*** Iter {iteration} - Key {i}: Hamming {hamm} ***\n")
                if hamm < self.best_hamming:
                    self.best_hamming = hamm
                    self.best_priv = priv_batch[i].hex()
                    print(f"[{time.strftime('%H:%M:%S')}] New Best: Hamming {hamm}")
            sys.stdout.flush()

        sys.stdout.write("*** Done after 3 iterations ***\n")
        sys.stdout.flush()

ai = BitflipAI(BASE_PRIV, get_hash160_diff_bits(BASE_HASH160, TARGET_HASH160_BYTES))
ai.run()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.3/149.3 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 109.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 89.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu

In [2]:
priv_bytes = bytes.fromhex("a897944d9ff3a7cc2805ad01029b5026671ff706db177b10dbc0d0f3962983f3")
sk = SigningKey.from_string(priv_bytes, curve=SECP256k1)
pub = b'\04' + sk.get_verifying_key().to_string()
sha = hashlib.sha256(pub).digest()
ripemd = RIPEMD160.new()
ripemd.update(sha)
ripemd_digest = ripemd.digest()  # This is the Hash160
print(ripemd_digest.hex())  # Outputs: "9d27ea8eabf6eecf07f7e6c9ae88c39f1aead750"

99b0890eb79157b4f9fada52020a6326619d8fd2


In [3]:
target = bytes.fromhex("a0b0d60e5991578ed37cbda2b17d8b2ce23ab295")
new_hash = bytes.fromhex("99b0890eb79157b4f9fada52020a6326619d8fd2")
hamm = np.sum(np.frombuffer(target, dtype=np.uint8) != np.frombuffer(new_hash, dtype=np.uint8))
print(hamm)  # Outputs: 16

16


In [4]:
target = bytes.fromhex("a0b0d60e5991578ed37cbda2b17d8b2ce23ab295")
new_hash = bytes.fromhex("99b0890eb79157b4f9fada52020a6326619d8fd2")
hamm = np.sum(np.frombuffer(target, dtype=np.uint8) != np.frombuffer(new_hash, dtype=np.uint8))
print(hamm)  # 16

16


In [5]:
target = bytes.fromhex("a0b0d60e5991578ed37cbda2b17d8b2ce23ab295")
new_hash = bytes.fromhex("99b0890eb79157b4f9fada52020a6326619d8fd2")
hamm = np.sum(np.frombuffer(target, dtype=np.uint8) != np.frombuffer(new_hash, dtype=np.uint8))
print(hamm)  # 16

16


In [6]:
target = bytes.fromhex("a0b0d60e5991578ed37cbda2b17d8b2ce23ab295")
new_hash = bytes.fromhex("99b0890eb79157b4f9fada52020a6326619d8fd2")
hamm = np.sum(np.frombuffer(target, dtype=np.uint8) != np.frombuffer(new_hash, dtype=np.uint8))
print(hamm)  # 16

16


In [7]:
!pip install pycryptodome ecdsa base58 numpy torch

import hashlib
from Crypto.Hash import RIPEMD160
from ecdsa import SigningKey, SECP256k1
import base58
import numpy as np
import random
import time
import sys

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
TARGET_HASH160_BYTES = bytes.fromhex("a0b0d60e5991578ed37cbda2b17d8b2ce23ab295")

BASES = [
    ("8b2aca2022fdaffad19df9bfc576871af60841ef526f9a7b5fcfb5923ab98f8d", "a0a8d608596e57bdfd53502ead468974df73f63c", "Original"),
    ("a897944d9ff3a7cc2805ad01029b5026671ff706db177b10dbc0d0f3962983f3", "9d27ea8eabf6eecf07f7e6c9ae88c39f1aead750", "Seed Scan 16"),
    ("fefa646609df29176194d80c380f9efa16253ac5510c9829dcac74418e18d5c2", "d3ed2eecf8b429eabf5cfae38aead750c0f8c39f", "Seed Scan 17"),
]

def derive_address_batch(priv_batch):
    results = []
    for priv_bytes in priv_batch:
        sk = SigningKey.from_string(priv_bytes, curve=SECP256k1)
        pub = b'\04' + sk.get_verifying_key().to_string()
        sha = hashlib.sha256(pub).digest()
        ripemd = RIPEMD160.new()
        ripemd.update(sha)
        ripemd_digest = ripemd.digest()
        payload = b'\x00' + ripemd_digest
        checksum = hashlib.sha256(hashlib.sha256(payload).digest()).digest()[:4]
        addr = base58.b58encode(payload + checksum).decode()
        results.append((addr, ripemd_digest.hex()))
    return results

def hamming_distance(a, b):
    return np.sum(np.frombuffer(a, dtype=np.uint8) != np.frombuffer(b, dtype=np.uint8))

def flip_bit_batch(base_priv, flipped_bits_batch):
    priv_batch = []
    for flipped_bits in flipped_bits_batch:
        priv_array = bytearray(base_priv)
        for bit_pos in flipped_bits:
            byte_idx = bit_pos // 8
            bit_idx = bit_pos % 8
            priv_array[byte_idx] ^= (1 << bit_idx)
        priv_batch.append(bytes(priv_array))
    return priv_batch

def get_hash160_diff_bits(base_h160, target_h160):
    diff_bits = []
    for byte_idx, (b, t) in enumerate(zip(base_h160, target_h160)):
        xor = b ^ t
        for bit_idx in range(8):
            if xor & (1 << bit_idx):
                diff_bits.append(byte_idx * 8 + bit_idx)
    return diff_bits[:16]

class BitflipAI:
    def __init__(self, base_priv, base_hash160, label):
        self.base_priv = bytes.fromhex(base_priv)
        self.best_priv = base_priv
        self.best_addr = None
        self.best_hash160 = base_hash160
        self.best_hamming = hamming_distance(bytes.fromhex(base_hash160), TARGET_HASH160_BYTES)
        self.diff_bits = get_hash160_diff_bits(bytes.fromhex(base_hash160), TARGET_HASH160_BYTES)
        self.label = label

    def get_action(self, state):
        actions = [0] * 256
        for bit in self.diff_bits:
            if random.random() < 0.5:  # 50% flip chance for control
                actions[bit] = 1
        for bit in range(256):
            if random.random() < 0.05:  # 5% random flips
                actions[bit] = 1 if actions[bit] == 0 else 0
        return actions

    def run(self):
        print(f"\nStarting with {self.label}: Priv {self.best_priv[:16]}..., Hamming {self.best_hamming}")
        batch_size = 64
        iteration = 0

        while iteration < 50:  # Short run
            iteration += 1
            state = bytes.fromhex(self.best_priv)
            actions_batch = [self.get_action(state) for _ in range(batch_size)]
            flipped_bits_batch = [[i for i, a in enumerate(actions) if a == 1] for actions in actions_batch]
            priv_batch = flip_bit_batch(state, flipped_bits_batch)

            addr_hash160_batch = derive_address_batch(priv_batch)
            for i, (addr, hash160) in enumerate(addr_hash160_batch):
                hamm = hamming_distance(bytes.fromhex(hash160), TARGET_HASH160_BYTES)
                if hamm < self.best_hamming:
                    self.best_hamming = hamm
                    self.best_priv = priv_batch[i].hex()
                    self.best_addr = addr
                    self.best_hash160 = hash160
                    print(f"[{time.strftime('%H:%M:%S')}] New Best (Iter {iteration}):")
                    print(f"  Priv: {self.best_priv}")
                    print(f"  Addr: {self.best_addr}")
                    print(f"  Hamming: {self.best_hamming}")
                if hamm == 0:
                    print(f"[{time.strftime('%H:%M:%S')}] JACKPOT! Hamming 0 Found!")
                    return

        print(f"Finished with {self.label}: Hamming {self.best_hamming}")

# Run for each base
for base_priv, base_hash160, label in BASES:
    ai = BitflipAI(base_priv, base_hash160, label)
    ai.run()

Using device: cuda

Starting with Original: Priv 8b2aca2022fdaffa..., Hamming 16
Finished with Original: Hamming 16

Starting with Seed Scan 16: Priv a897944d9ff3a7cc..., Hamming 20
[22:42:33] New Best (Iter 1):
  Priv: a092a0d997f3afcc2803ad01829b5822651ff706db175b11d3c0b0f3966983f3
  Addr: 1MuXx2oKqCzQtkAiLut18zGTvYdXUn5J9A
  Hamming: 19
[22:42:34] New Best (Iter 4):
  Priv: a982be5997f3adcc2c13a541829b7822551ef706d9178b13d1c0b0f3966983f2
  Addr: 1C9HSgn4ev7zcoCvqUAPZyh751kMqM6C1o
  Hamming: 18
Finished with Seed Scan 16: Hamming 18

Starting with Seed Scan 17: Priv fefa646609df2917..., Hamming 20
[22:42:36] New Best (Iter 1):
  Priv: ffe6746689dfa0376194d808380f8efa16a53ac5500c9821dcac76418e18d5c2
  Addr: 1AzrtGycUCPHUUWurvzDqwGnDpb7QMdFht
  Hamming: 19
[22:42:37] New Best (Iter 23):
  Priv: fdb33c6689dfa0376194d868380f86fa16a53ac1d04c9827dcac7641ce10d5e2
  Addr: 1MoAiCKR5yNXYNJYy4GepzpCp47oHPFaA7
  Hamming: 18
Finished with Seed Scan 17: Hamming 18


In [8]:
!pip install pycryptodome ecdsa base58 numpy torch

import hashlib
from Crypto.Hash import RIPEMD160
from ecdsa import SigningKey, SECP256k1
import base58
import numpy as np
import random
import time
import sys

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
TARGET_HASH160_BYTES = bytes.fromhex("a0b0d60e5991578ed37cbda2b17d8b2ce23ab295")

BASES = [
    ("8b2aca2022fdaffad19df9bfc576871af60841ef526f9a7b5fcfb5923ab98f8d", "a0a8d608596e57bdfd53502ead468974df73f63c", "Original"),
    ("a897944d9ff3a7cc2805ad01029b5026671ff706db177b10dbc0d0f3962983f3", "9d27ea8eabf6eecf07f7e6c9ae88c39f1aead750", "Seed Scan 16"),
    ("fefa646609df29176194d80c380f9efa16253ac5510c9829dcac74418e18d5c2", "d3ed2eecf8b429eabf5cfae38aead750c0f8c39f", "Seed Scan 17"),
]

def derive_address_batch(priv_batch):
    results = []
    for priv_bytes in priv_batch:
        sk = SigningKey.from_string(priv_bytes, curve=SECP256k1)
        pub = b'\04' + sk.get_verifying_key().to_string()
        sha = hashlib.sha256(pub).digest()
        ripemd = RIPEMD160.new()
        ripemd.update(sha)
        ripemd_digest = ripemd.digest()
        payload = b'\x00' + ripemd_digest
        checksum = hashlib.sha256(hashlib.sha256(payload).digest()).digest()[:4]
        addr = base58.b58encode(payload + checksum).decode()
        results.append((addr, ripemd_digest.hex()))
    return results

def hamming_distance(a, b):
    return np.sum(np.frombuffer(a, dtype=np.uint8) != np.frombuffer(b, dtype=np.uint8))

def flip_bit_batch(base_priv, flipped_bits_batch):
    priv_batch = []
    for flipped_bits in flipped_bits_batch:
        priv_array = bytearray(base_priv)
        for bit_pos in flipped_bits:
            byte_idx = bit_pos // 8
            bit_idx = bit_pos % 8
            priv_array[byte_idx] ^= (1 << bit_idx)
        priv_batch.append(bytes(priv_array))
    return priv_batch

def get_hash160_diff_bits(base_h160, target_h160):
    diff_bits = []
    for byte_idx, (b, t) in enumerate(zip(base_h160, target_h160)):
        xor = b ^ t
        for bit_idx in range(8):
            if xor & (1 << bit_idx):
                diff_bits.append(byte_idx * 8 + bit_idx)
    return diff_bits[:16]

class BitflipAI:
    def __init__(self, base_priv, base_hash160, label):
        self.base_priv = bytes.fromhex(base_priv)
        self.best_priv = base_priv
        self.best_addr = None
        self.best_hash160 = base_hash160
        self.best_hamming = hamming_distance(bytes.fromhex(base_hash160), TARGET_HASH160_BYTES)
        self.diff_bits = get_hash160_diff_bits(bytes.fromhex(base_hash160), TARGET_HASH160_BYTES)
        self.label = label

    def get_action(self, state):
        actions = [0] * 256
        for bit in self.diff_bits:
            if random.random() < 0.5:  # 50% flip chance
                actions[bit] = 1
        for bit in range(256):
            if random.random() < 0.05:
                actions[bit] = 1 if actions[bit] == 0 else 0
        return actions

    def run(self):
        print(f"\nStarting with {self.label}: Priv {self.best_priv[:16]}..., Hamming {self.best_hamming}")
        batch_size = 64
        iteration = 0

        while iteration < 50:
            iteration += 1
            state = bytes.fromhex(self.best_priv)
            actions_batch = [self.get_action(state) for _ in range(batch_size)]
            flipped_bits_batch = [[i for i, a in enumerate(actions) if a == 1] for actions in actions_batch]
            priv_batch = flip_bit_batch(state, flipped_bits_batch)

            addr_hash160_batch = derive_address_batch(priv_batch)
            for i, (addr, hash160) in enumerate(addr_hash160_batch):
                hamm = hamming_distance(bytes.fromhex(hash160), TARGET_HASH160_BYTES)
                if hamm < self.best_hamming:
                    self.best_hamming = hamm
                    self.best_priv = priv_batch[i].hex()
                    self.best_addr = addr
                    self.best_hash160 = hash160
                    print(f"[{time.strftime('%H:%M:%S')}] New Best (Iter {iteration}):")
                    print(f"  Priv: {self.best_priv}")
                    print(f"  Addr: {self.best_addr}")
                    print(f"  Hamming: {self.best_hamming}")
                if hamm == 0:
                    print(f"[{time.strftime('%H:%M:%S')}] JACKPOT! Hamming 0 Found!")
                    return

        print(f"Finished with {self.label}: Hamming {self.best_hamming}")

for base_priv, base_hash160, label in BASES:
    ai = BitflipAI(base_priv, base_hash160, label)
    ai.run()

Using device: cuda

Starting with Original: Priv 8b2aca2022fdaffa..., Hamming 16
Finished with Original: Hamming 16

Starting with Seed Scan 16: Priv a897944d9ff3a7cc..., Hamming 20
[22:44:32] New Best (Iter 1):
  Priv: a112b04f9df3a74c2805ac01028b5026671ff706db177b10dbe0d0f3920983f3
  Addr: 19ciUWQU6em9iaiETNbBWNaUHru7UzZymT
  Hamming: 19
[22:44:33] New Best (Iter 4):
  Priv: ad87b0cf9ff3a75c2805ec01028bd026671ff706c9157b10dbe0d8b3920986f3
  Addr: 17P7iwc3UBPYeKzsG72SZWxEJFgFLMAcak
  Hamming: 18
[22:44:33] New Best (Iter 11):
  Priv: b185904f9ff3a7dc2805e6d1028bd02e271ff706e1557b12d3e0d8b3924986f3
  Addr: 137td4ZtHQg2yfhHTkZ8dkPkd3kfVvcVsa
  Hamming: 17
Finished with Seed Scan 16: Hamming 17

Starting with Seed Scan 17: Priv fefa646609df2917..., Hamming 20
[22:44:35] New Best (Iter 1):
  Priv: dea9fc6449df291763f65888384f9ffe16b53ac5510cd829dc2c7c418e18ddc2
  Addr: 18bcTfeSW68cJ1WE5oZGVu7bdyoJUD16V
  Hamming: 19
[22:44:35] New Best (Iter 2):
  Priv: c4f45c6649df291763f7d088686d9fbe06b

In [1]:
!pip install pycryptodome ecdsa base58 numpy torch

import hashlib
from Crypto.Hash import RIPEMD160
from ecdsa import SigningKey, SECP256k1
import base58
import numpy as np
import random
import time
import sys

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
TARGET_HASH160_BYTES = bytes.fromhex("a0b0d60e5991578ed37cbda2b17d8b2ce23ab295")

BASES = [
    ("8b2aca2022fdaffad19df9bfc576871af60841ef526f9a7b5fcfb5923ab98f8d", "a0a8d608596e57bdfd53502ead468974df73f63c", "Original"),
    ("a897944d9ff3a7cc2805ad01029b5026671ff706db177b10dbc0d0f3962983f3", "9d27ea8eabf6eecf07f7e6c9ae88c39f1aead750", "Seed Scan 16"),
    ("fefa646609df29176194d80c380f9efa16253ac5510c9829dcac74418e18d5c2", "d3ed2eecf8b429eabf5cfae38aead750c0f8c39f", "Seed Scan 17"),
]

def derive_address_batch(priv_batch):
    results = []
    for priv_bytes in priv_batch:
        sk = SigningKey.from_string(priv_bytes, curve=SECP256k1)
        pub = b'\04' + sk.get_verifying_key().to_string()
        sha = hashlib.sha256(pub).digest()
        ripemd = RIPEMD160.new()
        ripemd.update(sha)
        ripemd_digest = ripemd.digest()
        payload = b'\x00' + ripemd_digest
        checksum = hashlib.sha256(hashlib.sha256(payload).digest()).digest()[:4]
        addr = base58.b58encode(payload + checksum).decode()
        results.append((addr, ripemd_digest.hex()))
    return results

def hamming_distance(a, b):
    return np.sum(np.frombuffer(a, dtype=np.uint8) != np.frombuffer(b, dtype=np.uint8))

def flip_bit_batch(base_priv, flipped_bits_batch):
    priv_batch = []
    for flipped_bits in flipped_bits_batch:
        priv_array = bytearray(base_priv)
        for bit_pos in flipped_bits:
            byte_idx = bit_pos // 8
            bit_idx = bit_pos % 8
            priv_array[byte_idx] ^= (1 << bit_idx)
        priv_batch.append(bytes(priv_array))
    return priv_batch

def get_hash160_diff_bits(base_h160, target_h160):
    diff_bits = []
    for byte_idx, (b, t) in enumerate(zip(base_h160, target_h160)):
        xor = b ^ t
        for bit_idx in range(8):
            if xor & (1 << bit_idx):
                diff_bits.append(byte_idx * 8 + bit_idx)
    return diff_bits[:16]

class BitflipAI:
    def __init__(self, base_priv, base_hash160, label):
        self.base_priv = bytes.fromhex(base_priv)
        self.best_priv = base_priv
        self.best_addr = None
        self.best_hash160 = base_hash160
        self.best_hamming = hamming_distance(bytes.fromhex(base_hash160), TARGET_HASH160_BYTES)
        self.diff_bits = get_hash160_diff_bits(bytes.fromhex(base_hash160), TARGET_HASH160_BYTES)
        self.label = label

    def get_action(self, state):
        actions = [0] * 256
        for bit in self.diff_bits:
            if random.random() < 0.5:  # 50% flip chance
                actions[bit] = 1
        for bit in range(256):
            if random.random() < 0.05:
                actions[bit] = 1 if actions[bit] == 0 else 0
        return actions

    def run(self):
        print(f"\nStarting with {self.label}: Priv {self.best_priv[:16]}..., Hamming {self.best_hamming}")
        batch_size = 64
        iteration = 0

        while iteration < 50:
            iteration += 1
            state = bytes.fromhex(self.best_priv)
            actions_batch = [self.get_action(state) for _ in range(batch_size)]
            flipped_bits_batch = [[i for i, a in enumerate(actions) if a == 1] for actions in actions_batch]
            priv_batch = flip_bit_batch(state, flipped_bits_batch)

            addr_hash160_batch = derive_address_batch(priv_batch)
            for i, (addr, hash160) in enumerate(addr_hash160_batch):
                hamm = hamming_distance(bytes.fromhex(hash160), TARGET_HASH160_BYTES)
                if hamm < self.best_hamming:
                    self.best_hamming = hamm
                    self.best_priv = priv_batch[i].hex()
                    self.best_addr = addr
                    self.best_hash160 = hash160
                    print(f"[{time.strftime('%H:%M:%S')}] New Best (Iter {iteration}):")
                    print(f"  Priv: {self.best_priv}")
                    print(f"  Addr: {self.best_addr}")
                    print(f"  Hamming: {self.best_hamming}")
                if hamm == 0:
                    print(f"[{time.strftime('%H:%M:%S')}] JACKPOT! Hamming 0 Found!")
                    return

        print(f"Finished with {self.label}: Hamming {self.best_hamming}")

for base_priv, base_hash160, label in BASES:
    ai = BitflipAI(base_priv, base_hash160, label)
    ai.run()

NameError: name 'torch' is not defined

In [2]:
!pip install pycryptodome ecdsa base58 numpy torch

In [1]:
import hashlib
from Crypto.Hash import RIPEMD160
from ecdsa import SigningKey, SECP256k1
import base58
import numpy as np
import torch
import random
import time
import sys

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
TARGET_HASH160_BYTES = bytes.fromhex("a0b0d60e5991578ed37cbda2b17d8b2ce23ab295")

BASES = [
    ("8b2aca2022fdaffad19df9bfc576871af60841ef526f9a7b5fcfb5923ab98f8d", "a0a8d608596e57bdfd53502ead468974df73f63c", "Original"),
    ("a897944d9ff3a7cc2805ad01029b5026671ff706db177b10dbc0d0f3962983f3", "9d27ea8eabf6eecf07f7e6c9ae88c39f1aead750", "Seed Scan 16"),
    ("fefa646609df29176194d80c380f9efa16253ac5510c9829dcac74418e18d5c2", "d3ed2eecf8b429eabf5cfae38aead750c0f8c39f", "Seed Scan 17"),
]

def derive_address_batch(priv_batch):
    results = []
    for priv_bytes in priv_batch:
        sk = SigningKey.from_string(priv_bytes, curve=SECP256k1)
        pub = b'\04' + sk.get_verifying_key().to_string()
        sha = hashlib.sha256(pub).digest()
        ripemd = RIPEMD160.new()
        ripemd.update(sha)
        ripemd_digest = ripemd.digest()
        payload = b'\x00' + ripemd_digest
        checksum = hashlib.sha256(hashlib.sha256(payload).digest()).digest()[:4]
        addr = base58.b58encode(payload + checksum).decode()
        results.append((addr, ripemd_digest.hex()))
    return results

def hamming_distance(a, b):
    return np.sum(np.frombuffer(a, dtype=np.uint8) != np.frombuffer(b, dtype=np.uint8))

def flip_bit_batch(base_priv, flipped_bits_batch):
    priv_batch = []
    for flipped_bits in flipped_bits_batch:
        priv_array = bytearray(base_priv)
        for bit_pos in flipped_bits:
            byte_idx = bit_pos // 8
            bit_idx = bit_pos % 8
            priv_array[byte_idx] ^= (1 << bit_idx)
        priv_batch.append(bytes(priv_array))
    return priv_batch

def get_hash160_diff_bits(base_h160, target_h160):
    diff_bits = []
    for byte_idx, (b, t) in enumerate(zip(base_h160, target_h160)):
        xor = b ^ t
        for bit_idx in range(8):
            if xor & (1 << bit_idx):
                diff_bits.append(byte_idx * 8 + bit_idx)
    return diff_bits[:16]

class BitflipAI:
    def __init__(self, base_priv, base_hash160, label):
        self.base_priv = bytes.fromhex(base_priv)
        self.best_priv = base_priv
        self.best_addr = None
        self.best_hash160 = base_hash160
        self.best_hamming = hamming_distance(bytes.fromhex(base_hash160), TARGET_HASH160_BYTES)
        self.diff_bits = get_hash160_diff_bits(bytes.fromhex(base_hash160), TARGET_HASH160_BYTES)
        self.label = label

    def get_action(self, state):
        actions = [0] * 256
        for bit in self.diff_bits:
            if random.random() < 0.5:  # 50% flip chance
                actions[bit] = 1
        for bit in range(256):
            if random.random() < 0.05:
                actions[bit] = 1 if actions[bit] == 0 else 0
        return actions

    def run(self):
        print(f"\nStarting with {self.label}: Priv {self.best_priv[:16]}..., Hamming {self.best_hamming}")
        batch_size = 64
        iteration = 0

        start_time = time.time()
        while iteration < 50:
            iteration += 1
            state = bytes.fromhex(self.best_priv)
            actions_batch = [self.get_action(state) for _ in range(batch_size)]
            flipped_bits_batch = [[i for i, a in enumerate(actions) if a == 1] for actions in actions_batch]
            priv_batch = flip_bit_batch(state, flipped_bits_batch)

            addr_hash160_batch = derive_address_batch(priv_batch)
            for i, (addr, hash160) in enumerate(addr_hash160_batch):
                hamm = hamming_distance(bytes.fromhex(hash160), TARGET_HASH160_BYTES)
                if hamm < self.best_hamming:
                    self.best_hamming = hamm
                    self.best_priv = priv_batch[i].hex()
                    self.best_addr = addr
                    self.best_hash160 = hash160
                    print(f"[{time.strftime('%H:%M:%S')}] New Best (Iter {iteration}):")
                    print(f"  Priv: {self.best_priv}")
                    print(f"  Addr: {self.best_addr}")
                    print(f"  Hamming: {self.best_hamming}")
                if hamm == 0:
                    print(f"[{time.strftime('%H:%M:%S')}] JACKPOT! Hamming 0 Found!")
                    return

        elapsed = time.time() - start_time
        print(f"Finished with {self.label}: Hamming {self.best_hamming}, Time: {elapsed:.2f}s")

for base_priv, base_hash160, label in BASES:
    ai = BitflipAI(base_priv, base_hash160, label)
    ai.run()

Using device: cuda
GPU Name: Tesla T4

Starting with Original: Priv 8b2aca2022fdaffa..., Hamming 16
Finished with Original: Hamming 16, Time: 2.43s

Starting with Seed Scan 16: Priv a897944d9ff3a7cc..., Hamming 20
[22:48:21] New Best (Iter 1):
  Priv: a11688ed99f2afcc2805ad01029b5026251ff786db377b51dbc0d0f7962982f3
  Addr: 18BDwEBwdoRxG2FQvkU3WXjXvGobkywjUr
  Hamming: 19
[22:48:21] New Best (Iter 1):
  Priv: 8187b4499ff2a7cc2815af01829bd026771fb706d9177b10db50d063962b83f3
  Addr: 1BmLSwr1DTf7bD7wUMKomyigdoMV4nR43c
  Hamming: 18
Finished with Seed Scan 16: Hamming 18, Time: 2.34s

Starting with Seed Scan 17: Priv fefa646609df2917..., Hamming 20
[22:48:24] New Best (Iter 1):
  Priv: ccba3c6409db29176196d81c380f9efb16253ac5510c9869dcac74c18e18d5c2
  Addr: 197xUcFheDaJraCHpp1u8RYSCF1QTynzMn
  Hamming: 19
[22:48:24] New Best (Iter 4):
  Priv: a4eb9c6409db29166196f81c580f9efb16253ac1514c9849ccac74c19a18d5c3
  Addr: 18p1ua5eF7e4VHYoMddTQgbXMpqXo832D1
  Hamming: 18
[22:48:25] New Best (Iter 28

In [2]:
self.best_hamming = hamming_distance(bytes.fromhex(base_hash160), TARGET_HASH160_BYTES)

NameError: name 'self' is not defined